# Reanalysis of NSCLC spatial-zone radiomics (Supplementary Table S3 version)

This notebook recalculates feature selection, final Cox models, and C-indices for max5, max10, and max20. Original-GTV radiomics are obtained from Le et al. Supplementary Tables S1, S2, and S3.

## 1. Locate the reanalysis package

Place the `colab_reanalysis` folder under `/content` in Google Colab. The notebook also detects an adjacent local folder.

In [ ]:
import os
from pathlib import Path

try:
    from google.colab import files
except ImportError:
    files = None

In [ ]:
candidates = [
    Path('/content/colab_reanalysis'),
    Path.cwd() / 'colab_reanalysis',
    Path('/workspace/tmp/colab_reanalysis'),
]
ROOT = next((path.resolve() for path in candidates if path.is_dir()), None)
if ROOT is None:
    raise FileNotFoundError('colab_reanalysis folder was not found. Upload it under /content or place it next to this notebook.')
print(f'Using package: {ROOT}')

In [ ]:
readme = ROOT / 'README.md'
assert readme.is_file(), readme
print(readme.read_text(encoding='utf-8').splitlines()[0])

In [ ]:
print('Package contents:')
for path in sorted(ROOT.iterdir()):
    print(' -', path.name)

In [ ]:
os.chdir(ROOT)
print(f'Working directory: {Path.cwd()}')

## 2. Install dependencies

In [ ]:
%pip install -q -r requirements-colab.txt

## 3. Verify the input files

In [ ]:
required = [
    ROOT / 'data/clinical_outcome/Supplementary-Table-S1-Training-set-LUNG1-Radiomics.csv',
    ROOT / 'data/clinical_outcome/Supplementary-Table-S2-Testing-set-LUNG1-Radiomics.csv',
    ROOT / 'data/clinical_outcome/Supplementary-Table-S3-Validation-set-LUNG2-Radiogenomics.csv',
    ROOT / 'data/common_cohort_manifest/aligned_case_manifest.csv',
    ROOT / 'data/spatial_features/features_physical2mm.csv',
    ROOT / 'data/spatial_features/features_physical3mm.csv',
    ROOT / 'data/spatial_features/features_physical5mm.csv',
]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError('Missing files:\n' + '\n'.join(missing))
print(f'Input-file check completed: {len(required)} files')

## 4. Recalculate max5, max10, and max20

The same five-fold cross-validation is run for each maximum feature count, so this step may take several minutes.

In [ ]:
import os
import subprocess
import sys

script = ROOT / 'reanalyze_strict_common_portable.py'
for cap in (5, 10, 20):
    output = ROOT / 'results' / f'max{cap}'
    output.mkdir(parents=True, exist_ok=True)
    env = os.environ.copy()
    env['MAX_FEATURES'] = str(cap)
    env['STRICT_OUT'] = str(output)
    env['REANALYSIS_DATA'] = str(ROOT / 'data')
    print(f'Running max{cap} ...', flush=True)
    subprocess.run([sys.executable, str(script)], env=env, check=True)
print('All analyses have completed.')

## 5. Compare the recalculated and S3-based reference results

In [ ]:
import numpy as np
import pandas as pd

def compare_cap(cap):
    generated = ROOT / 'results' / f'max{cap}'
    reference = ROOT / 'reference_results' / f'max{cap}'

    p_new = pd.read_csv(generated / 'performance.csv', encoding='utf-8-sig').sort_values(['condition', 'model', 'cohort']).reset_index(drop=True)
    p_ref = pd.read_csv(reference / 'performance.csv', encoding='utf-8-sig').sort_values(['condition', 'model', 'cohort']).reset_index(drop=True)
    performance_structure = p_new.drop(columns='c_index').equals(p_ref.drop(columns='c_index'))
    performance_numeric = np.allclose(p_new['c_index'], p_ref['c_index'], rtol=1e-10, atol=1e-12)

    s_new = pd.read_csv(generated / 'selected_features.csv', encoding='utf-8-sig')
    s_ref = pd.read_csv(reference / 'selected_features.csv', encoding='utf-8-sig')
    selected_equal = s_new.equals(s_ref)

    t_new = pd.read_csv(generated / 'elastic_net_tuning_summary.csv', encoding='utf-8-sig')
    t_ref = pd.read_csv(reference / 'elastic_net_tuning_summary.csv', encoding='utf-8-sig')
    tuning_structure = t_new.drop(columns='alpha').equals(t_ref.drop(columns='alpha'))
    tuning_numeric = np.allclose(t_new['alpha'], t_ref['alpha'], rtol=1e-12, atol=1e-14)

    return {
        'max_features': cap,
        'performance_equal': performance_structure and performance_numeric,
        'selected_features_equal': selected_equal,
        'tuning_equal': tuning_structure and tuning_numeric,
        'max_cindex_abs_diff': float(np.max(np.abs(p_new['c_index'] - p_ref['c_index']))),
    }

verification = pd.DataFrame([compare_cap(cap) for cap in (5, 10, 20)])
display(verification)
if not verification[['performance_equal', 'selected_features_equal', 'tuning_equal']].all().all():
    raise AssertionError('The recalculated results differ from the originals. Check the package versions and output files.')
print('The principal results match the original analysis.')

## 6. Display the external-validation results

In [ ]:
tables = []
for cap in (5, 10, 20):
    table = pd.read_csv(ROOT / 'results' / f'max{cap}' / 'performance.csv', encoding='utf-8-sig')
    table = table.loc[table['cohort'].eq('LUNG2_validation'), ['condition', 'model', 'n', 'events', 'c_index']].copy()
    table.insert(0, 'max_features', cap)
    tables.append(table)
display(pd.concat(tables, ignore_index=True))

## 7. Download the recalculated results

Optional

In [ ]:
# Optional in Google Colab:
# import shutil
# archive = shutil.make_archive('/content/reanalysis_results_s3', 'zip', ROOT / 'results')
# if files is not None:
#     files.download(archive)